# MNIST MLP (HEX export) 中間結果デバッグ（1枚）
- 入力: u8 hex (784 lines)
- 重み: row-major int8 hex
- bias: int32 hex
- 量子化: meta.json の in_zp/out_zp/rq_s/rq_shift/is_relu_fused/out_scale

# 目的:
1) 乗算 (xc*w)
2) 累算 Σ
3) bias加算
4) requant (half-up)
5) +out_zp
6) Clip (relu_fusedなら下限=out_zp)
7) u8 出力


In [ ]:
# # ============================================================
# # mlp_dump_lib.py (functions)  [HW-friendly arithmetic]
# # - mul:   u8 x i8  (8-bit operands; product computed in int16)
# # - acc:   int32
# # - +bias: int32 with SATURATING add (= clamp to int32)  <-- important
# # - requant (linear): int64 (mul + rounding + arithmetic shift) -> int32
# # - output: +out_zp then clip to u8 (relu_fused aware)
# #
# # - reverse32: display only (lane order unchanged internally)
# # ============================================================
# from __future__ import annotations
# from pathlib import Path
# import json
# import numpy as np

# PE_NUM = 32

# # clamp bounds (for clipping int64 -> int32)
# INT32_MIN_I64 = np.int64(-2**31)
# INT32_MAX_I64 = np.int64( 2**31 - 1)

# # ---------- file loaders ----------
# def load_meta(meta_path: Path) -> dict:
#     with meta_path.open("r", encoding="utf-8") as f:
#         return json.load(f)

# def load_u8_hex_n(path: Path, n: int) -> np.ndarray:
#     vals = []
#     with path.open("r", encoding="utf-8") as f:
#         for line in f:
#             s = line.strip()
#             if s:
#                 vals.append(int(s, 16) & 0xFF)
#     if len(vals) != n:
#         raise ValueError(f"input hex count mismatch: got {len(vals)} expected {n} ({path})")
#     return np.array(vals, dtype=np.uint8)

# def load_weight_rowmajor_int8(path: Path, out_ch: int, in_ch: int) -> np.ndarray:
#     vals = []
#     with path.open("r", encoding="utf-8") as f:
#         for line in f:
#             s = line.strip()
#             if s:
#                 vals.append(int(s, 16) & 0xFF)
#     exp = out_ch * in_ch
#     if len(vals) != exp:
#         raise ValueError(f"weight hex count mismatch: got {len(vals)} expected {exp} ({path})")
#     u = np.array(vals, dtype=np.uint8)
#     return u.view(np.int8).reshape(out_ch, in_ch)  # two's complement

# def load_bias_int32(path: Path, out_ch: int) -> np.ndarray:
#     vals = []
#     with path.open("r", encoding="utf-8") as f:
#         for line in f:
#             s = line.strip()
#             if s:
#                 vals.append(int(s, 16) & 0xFFFFFFFF)
#     if len(vals) != out_ch:
#         raise ValueError(f"bias hex count mismatch: got {len(vals)} expected {out_ch} ({path})")
#     u = np.array(vals, dtype=np.uint32)
#     return u.view(np.int32)

# # ---------- math helpers ----------
# def clamp_i32_from_i64_vec(x_i64: np.ndarray) -> np.ndarray:
#     """int64 -> int32 with saturation."""
#     return np.clip(x_i64, INT32_MIN_I64, INT32_MAX_I64).astype(np.int32)

# def sat_add_i32_vec(a_i32: np.ndarray, b_i32: np.ndarray) -> np.ndarray:
#     """
#     int32 saturating add (no wrap).
#     HW想定：acc(int32) + bias(int32) の結果を int32 範囲に clamp する。
#     """
#     a = a_i32.astype(np.int32, copy=False)
#     b = b_i32.astype(np.int32, copy=False)
#     s64 = a.astype(np.int64) + b.astype(np.int64)   # for detection only
#     return clamp_i32_from_i64_vec(s64)

# def requant_half_up_i32_vec_from_i32(acc_i32: np.ndarray, rq_s: int, rq_shift: int) -> np.ndarray:
#     """
#     Linear requant using 64-bit intermediate:
#       mul64 = (int64)acc_i32 * rq_s
#       rnd64 = mul64 + 2^(rq_shift-1)
#       out64 = rnd64 >> rq_shift (arithmetic)
#       clamp to int32
#     """
#     mul64 = acc_i32.astype(np.int64) * np.int64(rq_s)
#     if rq_shift <= 0:
#         return clamp_i32_from_i64_vec(mul64)
#     rnd64 = mul64 + (np.int64(1) << np.int64(rq_shift - 1))
#     out64 = rnd64 >> np.int64(rq_shift)  # arithmetic shift
#     return clamp_i32_from_i64_vec(out64)

# def clip_to_u8_vec(qy_i64: np.ndarray, out_zp: int, relu_fused: bool) -> np.ndarray:
#     lo = out_zp if relu_fused else 0
#     return np.clip(qy_i64, lo, 255).astype(np.uint8)

# # ---------- formatting helpers (HEX, no inner spaces) ----------
# def hex2_u8(v: int) -> str:
#     return f"{(v & 0xFF):02x}"

# def pack_u8_32(vec_u8: np.ndarray) -> str:
#     return "".join(f"{int(x) & 0xFF:02x}" for x in vec_u8)

# def pack_i8_32_as_u8hex(vec_i8: np.ndarray) -> str:
#     u8 = (vec_i8.astype(np.int32) & 0xFF).astype(np.uint8)
#     return pack_u8_32(u8)

# def pack_i32_32(vec_i32: np.ndarray) -> str:
#     u32 = vec_i32.astype(np.int32).view(np.uint32)
#     return "".join(f"{int(x):08x}" for x in u32)

# def ensure_len32(vec: np.ndarray, pad_value: int, dtype) -> np.ndarray:
#     out = np.full((PE_NUM,), pad_value, dtype=dtype)
#     n = min(PE_NUM, vec.shape[0])
#     out[:n] = vec[:n].astype(dtype, copy=False)
#     return out

# def _write_line(out_f, s: str):
#     if out_f is None:
#         print(s)
#     else:
#         out_f.write(s + "\n")

# # ---------- label helper ----------
# def make_step_tag(layer_name: str, group_idx_1based: int, group_cnt: int, k_1based: int, L: int) -> str:
#     fc = layer_name.replace("fc", "FC")
#     pad = len(str(L))
#     k_str = str(k_1based).zfill(pad)
#     return f"[{fc}({group_idx_1based}/{group_cnt}), acc({k_str}/{L})]"

# # ---------- display helper ----------
# def maybe_rev32(vec32: np.ndarray, reverse32: bool) -> np.ndarray:
#     return vec32[::-1] if reverse32 else vec32

# # ---------- core dump for one layer ----------
# def dump_layer_cyclewise(
#     layer_name: str,
#     x_u8: np.ndarray,     # [in_ch], uint8
#     layer_meta: dict,     # meta[layer_name]
#     W: np.ndarray,        # [out_ch, in_ch], int8
#     b: np.ndarray,        # [out_ch], int32
#     show: dict,           # keys: x,wvec,mul,acc,plus_bias,clip,quant, reverse32
#     k_range=None,         # None or range/list of k (0-based indices)
#     group_bases=None,     # None or list of out_base (0,32,64...)
#     out_f=None,
#     header=True,
# ):
#     out_ch, in_ch = map(int, layer_meta["w_shape"])
#     assert x_u8.shape[0] == in_ch, f"{layer_name}: x length mismatch {x_u8.shape[0]} vs {in_ch}"
#     assert W.shape == (out_ch, in_ch)
#     assert b.shape == (out_ch,)

#     in_zp  = int(layer_meta["in_zero_point"])
#     out_zp = int(layer_meta["out_zero_point"])
#     rq_s   = int(layer_meta["requant_s"])
#     rq_sh  = int(layer_meta["requant_shift"])
#     relu_fused = bool(layer_meta["is_relu_fused"])

#     reverse32 = bool(show.get("reverse32", False))

#     k_list = range(in_ch) if k_range is None else k_range

#     all_group_bases = list(range(0, out_ch, PE_NUM))
#     group_cnt = len(all_group_bases)
#     if group_bases is None:
#         group_bases = all_group_bases

#     if header:
#         _write_line(out_f, "")
#         _write_line(out_f, "#"*120)
#         _write_line(out_f, f"# LAYER={layer_name}  in_ch={in_ch} out_ch={out_ch} groups={all_group_bases}")
#         _write_line(out_f, f"# in_zp={in_zp} out_zp={out_zp} rq_s={rq_s} rq_shift={rq_sh} relu_fused={relu_fused}")
#         _write_line(out_f, f"# reverse32(display only) = {reverse32}")
#         _write_line(out_f, "# line tag format: [FC#(M/N), acc(K/L)]")
#         _write_line(out_f, "#"*120)

#     y_out = np.zeros((out_ch,), dtype=np.uint8)

#     for out_base in group_bases:
#         if out_base not in all_group_bases:
#             raise ValueError(f"{layer_name}: invalid out_base {out_base}, valid={all_group_bases}")

#         group_idx_1based = (out_base // PE_NUM) + 1
#         valid = min(PE_NUM, out_ch - out_base)

#         # bias padded to 32 lanes
#         b_vec = b[out_base:out_base+valid].astype(np.int32, copy=False)
#         b_pad = ensure_len32(b_vec, 0, np.int32)

#         # acc32 (HW spec)
#         acc32 = np.zeros((PE_NUM,), dtype=np.int32)
#         last_quant_u8 = np.zeros((PE_NUM,), dtype=np.uint8)

#         _write_line(out_f, "")
#         _write_line(out_f, f"# --- {layer_name} out_base={out_base} (group {group_idx_1based}/{group_cnt}, valid lanes={valid}) ---")

#         for k in k_list:
#             k1 = int(k) + 1
#             tag = make_step_tag(layer_name, group_idx_1based, group_cnt, k1, in_ch)

#             xk_u8 = int(x_u8[k])

#             # --- 8-bit operands for multiply ---
#             # input: uint8, weight: int8
#             # If in_zp != 0, we still do (u8 - in_zp) in int16 (HWなら前段で補正)。
#             x_center_i16 = np.int16(np.int16(x_u8[k]) - np.int16(in_zp))  # range typically within [-255,255]
#             w_vec_i8 = W[out_base:out_base+valid, k]                     # int8
#             w_pad_i8 = ensure_len32(w_vec_i8, 0, np.int8)

#             # mul: (int16)centered_x * (int16)w  -> int16 (safe range ~ +/-32640)
#             mul16 = (x_center_i16.astype(np.int16) * w_pad_i8.astype(np.int16)).astype(np.int16)

#             # acc: int32
#             acc32 = (acc32 + mul16.astype(np.int32)).astype(np.int32)

#             # +bias: int32 saturating (clamp included)
#             sum_sat_i32 = sat_add_i32_vec(acc32, b_pad)   # int32

#             # clip stage (already clamped)
#             clip_i32 = sum_sat_i32

#             # requant: int64 intermediate
#             rq_i32 = requant_half_up_i32_vec_from_i32(clip_i32, rq_s, rq_sh)

#             # add out_zp then output clip (use int64 for safety)
#             qy_i64 = rq_i32.astype(np.int64) + np.int64(out_zp)
#             q_u8 = clip_to_u8_vec(qy_i64, out_zp, relu_fused)
#             last_quant_u8 = q_u8

#             # ---- display (optionally reverse 32-lane vectors) ----
#             w_disp    = maybe_rev32(w_pad_i8.astype(np.int8, copy=False), reverse32)
#             mul_disp  = maybe_rev32(mul16.astype(np.int32, copy=False), reverse32)
#             acc_disp  = maybe_rev32(acc32, reverse32)
#             pb_disp   = maybe_rev32(sum_sat_i32, reverse32)
#             clip_disp = maybe_rev32(clip_i32, reverse32)
#             q_disp    = maybe_rev32(q_u8, reverse32)

#             parts = [tag]
#             if show.get("x", False):
#                 parts.append(hex2_u8(xk_u8))
#             if show.get("wvec", False):
#                 parts.append(pack_i8_32_as_u8hex(w_disp))
#             if show.get("mul", False):
#                 parts.append(pack_i32_32(mul_disp))
#             if show.get("acc", False):
#                 parts.append(pack_i32_32(acc_disp))
#             if show.get("plus_bias", False):
#                 parts.append(pack_i32_32(pb_disp))
#             if show.get("clip", False):
#                 parts.append(pack_i32_32(clip_disp))
#             if show.get("quant", False):
#                 parts.append(pack_u8_32(q_disp))

#             _write_line(out_f, " ".join(parts))

#         # final output of this group = quant at last k（内部のlane順は維持）
#         y_out[out_base:out_base+valid] = last_quant_u8[:valid]

#     return y_out

# # ---------- MLP runner (fc1->fc2->fc3) ----------
# def dump_mlp_cyclewise(
#     export_dir: Path,
#     input_hex: Path,
#     show: dict,
#     out_f=None,
#     k_range_override: dict | None = None,
#     group_base_override: dict | None = None,
#     expected_label: int | None = None,   # <-- add
# ):
#     meta = load_meta(export_dir / "meta.json")

#     x0_u8 = load_u8_hex_n(input_hex, 784)

#     def load_layer_params(name: str):
#         out_ch, in_ch = map(int, meta[name]["w_shape"])
#         W = load_weight_rowmajor_int8(export_dir / f"{name}_W_rowmajor_int8.hex", out_ch, in_ch)
#         b = load_bias_int32(export_dir / f"{name}_b_int32.hex", out_ch)
#         return W, b

#     W1, b1 = load_layer_params("fc1")
#     W2, b2 = load_layer_params("fc2")
#     W3, b3 = load_layer_params("fc3")

#     k_over = k_range_override or {}
#     g_over = group_base_override or {}

#     y1_u8 = dump_layer_cyclewise(
#         "fc1", x0_u8, meta["fc1"], W1, b1,
#         show=show,
#         k_range=k_over.get("fc1", None),
#         group_bases=g_over.get("fc1", None),
#         out_f=out_f,
#         header=True
#     )

#     y2_u8 = dump_layer_cyclewise(
#         "fc2", y1_u8, meta["fc2"], W2, b2,
#         show=show,
#         k_range=k_over.get("fc2", None),
#         group_bases=g_over.get("fc2", None),
#         out_f=out_f,
#         header=True
#     )

#     y3_u8 = dump_layer_cyclewise(
#         "fc3", y2_u8, meta["fc3"], W3, b3,
#         show=show,
#         k_range=k_over.get("fc3", None),
#         group_bases=g_over.get("fc3", None),
#         out_f=out_f,
#         header=True
#     )

#     pred = int(np.argmax(y3_u8[:10]))
#     _write_line(out_f, "")
#     _write_line(out_f, "="*120)
#     _write_line(out_f, f"# FINAL y3_u8 (0..9) = {[int(v) for v in y3_u8[:10]]}")
#     _write_line(out_f, f"# PRED = {pred}")
#     if expected_label is not None:
#         ok = (pred == int(expected_label))
#         _write_line(out_f, f"# EXPECTED_LABEL = {int(expected_label)}  -> {'OK' if ok else 'NG'}")
#     _write_line(out_f, "="*120)

#     return dict(y1=y1_u8, y2=y2_u8, y3=y3_u8, pred=pred)

In [ ]:
# # ============================================================
# # main (lightweight usage)
# # ============================================================
# from pathlib import Path

# # --- paths ---
# EXPORT_DIR = Path(r"../export_hw_fixed_halfup_rowmajor")
# INPUT_HEX  = EXPORT_DIR / "mnist_inputs_hex_split" / "correct" / "mnist_00000_label7_pred7_OK_u8.hex"

# # --- output destination ---
# OUT_TXT = None  # 例: Path("mlp_cycle_dump.txt")  # ファイル出力推奨（行が多い）

# # --- what to show (True/False) ---
# SHOW = dict(
#     reverse32   =True,
#     x           =False,
#     wvec        =False,
#     mul         =False,
#     acc         =False,
#     plus_bias   =True,
#     clip        =False,
#     quant       =False,
# )

# # --- optional: limit printing to make it lighter ---
# K_RANGE_OVERRIDE = {
#     "fc1": None,           # 例: range(0, 16)
#     "fc2": None,
#     "fc3": None,
# }

# # --- optional: choose which out_base groups to dump ---
# # fc1: [0,32] / fc2: [0] / fc3: [0]
# GROUP_BASE_OVERRIDE = {
#     "fc1": None,           # 例: [0] だけ
#     "fc2": None,
#     "fc3": None,
# }

# # --- run ---
# out_f = None
# if OUT_TXT is not None:
#     OUT_TXT.parent.mkdir(parents=True, exist_ok=True)
#     out_f = OUT_TXT.open("w", encoding="utf-8")

# try:
#     result = dump_mlp_cyclewise(
#         export_dir=EXPORT_DIR,
#         input_hex=INPUT_HEX,
#         show=SHOW,
#         out_f=out_f,
#         k_range_override=K_RANGE_OVERRIDE,
#         group_base_override=GROUP_BASE_OVERRIDE,
#     )
# finally:
#     if out_f is not None:
#         out_f.close()

# print("pred =", result["pred"])
# print("y3_u8 =", result["y3"][:10].tolist())



########################################################################################################################
# LAYER=fc1  in_ch=784 out_ch=64 groups=[0, 32]
# in_zp=0 out_zp=0 rq_s=423831 rq_shift=30 relu_fused=True
# reverse32(display only) = True
# line tag format: [FC#(M/N), acc(K/L)]
########################################################################################################################

# --- fc1 out_base=0 (group 1/2, valid lanes=32) ---
[FC1(1/2), acc(001/784)] 00001bbbfffff75f000003c8000010070000190200000d6000001020000003a900000e550000073f00002193fffff4fb00000443000015b00000049d00000c54000003230000061300001368000010bafffff29400000819000012290000165f0000059100000e650000285dffffff5000001dbe000009c50000294afffff7e6
[FC1(1/2), acc(002/784)] 00001bbbfffff75f000003c8000010070000190200000d6000001020000003a900000e550000073f00002193fffff4fb00000443000015b00000049d00000c54000003230000061300001368000010bafffff29400000819000012290000165f0000059100000e650000285d

In [ ]:
# from pathlib import Path
# import difflib
# import builtins

# def generate_html_diff(text1, text2):
#     d = difflib.HtmlDiff(wrapcolumn=80)
#     return d.make_file(
#         text1.splitlines(), text2.splitlines(),
#         fromdesc="result.txt", todesc="output.txt"
#     )

# diff_dir = Path.cwd() / "diff"            # cwd基準で固定
# p1 = (diff_dir / "result.txt").resolve()  # 絶対パス化
# p2 = (diff_dir / "output.txt").resolve()

# print("p1:", p1)
# print("p2:", p2)

# # Windowsでパス解釈が怪しい時の回避（長パス/再解析ポイントに強い）
# p1w = r"\\?\{}".format(str(p1))
# p2w = r"\\?\{}".format(str(p2))

# with builtins.open(p1w, "r", encoding="utf-8", errors="replace", newline="") as f1:
#     t1 = f1.read()
# with builtins.open(p2w, "r", encoding="utf-8", errors="replace", newline="") as f2:
#     t2 = f2.read()

# html_diff = generate_html_diff(t1, t2)

# out_html = (diff_dir / "diff.html").resolve()
# outw = r"\\?\{}".format(str(out_html))
# with builtins.open(outw, "w", encoding="utf-8", newline="") as fout:
#     fout.write(html_diff)

# print("OK ->", out_html)


p1: D:\vlsi\code8bit\py_impl\diff\result.txt
p2: D:\vlsi\code8bit\py_impl\diff\output.txt
OK -> D:\vlsi\code8bit\py_impl\diff\diff.html


# ROM

In [ ]:
from __future__ import annotations
from pathlib import Path
import json
import numpy as np

PE_NUM   = 32
ACC_SIZE = 32

# ------------------------------------------------------------
# loader（今までのスクリプトと同じ挙動）
# ------------------------------------------------------------
def load_meta(meta_path: Path) -> dict:
    with meta_path.open("r", encoding="utf-8") as f:
        return json.load(f)

def load_weight_rowmajor_int8(path: Path, out_ch: int, in_ch: int) -> np.ndarray:
    vals = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                vals.append(int(s, 16) & 0xFF)
    exp = out_ch * in_ch
    if len(vals) != exp:
        raise ValueError(f"weight hex count mismatch: got {len(vals)} expected {exp} ({path})")
    u = np.array(vals, dtype=np.uint8)
    return u.view(np.int8).reshape(out_ch, in_ch)  # two's complement

def load_bias_int32(path: Path, out_ch: int) -> np.ndarray:
    vals = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                vals.append(int(s, 16) & 0xFFFFFFFF)
    if len(vals) != out_ch:
        raise ValueError(f"bias hex count mismatch: got {len(vals)} expected {out_ch} ({path})")
    u = np.array(vals, dtype=np.uint32)
    return u.view(np.int32)

# ------------------------------------------------------------
# W1/W2/W3 用 ROM テーブル生成
# ------------------------------------------------------------
def gen_weight_rom_table(layer_name: str,
                         W_i8: np.ndarray,
                         in_ch: int,
                         out_ch: int,
                         out_path: Path) -> None:
    """
    rd_data は 256-bit (8*32) 想定。
    lane0 を rd_data[7:0] に置くため、hex は lanes[::-1] で連結。
    """
    W = W_i8

    if layer_name == "fc1":
        # depth = (FC1_N/PE_NUM) * FC0_N = 2 * 784 = 1568
        depth = (out_ch // PE_NUM) * in_ch
    elif layer_name in ("fc2", "fc3"):
        # fc2: depth=64 (in_ch), fc3: depth=32 (in_ch)
        depth = in_ch
    else:
        raise ValueError(f"unknown layer_name {layer_name}")

    with out_path.open("w", encoding="utf-8") as f:
        f.write("// Auto-generated ROM table for %s weights\n" % layer_name)
        f.write("// addr -> rd_data (one 32-lane vector per address)\n\n")

        for addr in range(depth):
            if layer_name == "fc1":
                # testbench:
                #   for (i group=0..1)
                #     for (j in_ch=0..783)
                #       for (k lane)
                #         w1_wr_data[8*k +: 8] = W[(i*PE_NUM+k), j]
                group = addr // in_ch
                k_in  = addr % in_ch
                out_base = group * PE_NUM

                lanes: list[int] = []
                for k in range(PE_NUM):
                    out_idx = out_base + k
                    if out_idx < out_ch:
                        v = int(W[out_idx, k_in])  # int8
                        lanes.append(v & 0xFF)     # 2's complement as u8
                    else:
                        lanes.append(0)

            elif layer_name == "fc2":
                # testbench:
                #   for (i in_ch=0..63)
                #     for (j lane=0..31)
                #       w2_wr_data[8*j +: 8] = W[j, i]
                k_in = addr
                lanes = []
                for out_idx in range(PE_NUM):
                    if out_idx < out_ch:
                        v = int(W[out_idx, k_in])
                        lanes.append(v & 0xFF)
                    else:
                        lanes.append(0)

            else:  # fc3
                # testbench:
                #   for (i in_ch=0..31)
                #     w3_wr_data = 0
                #     for (j out=0..9)
                #       w3_wr_data[8*j +: 8] = W[j, i]
                k_in = addr
                lanes = [0] * PE_NUM
                for out_idx in range(out_ch):  # 0..9
                    v = int(W[out_idx, k_in])
                    lanes[out_idx] = v & 0xFF

            # lane0 を LSB に置くため逆順で連結（[31]..[0]）
            hex_data = "".join(f"{b:02x}" for b in lanes[::-1])
            f.write(f"        {addr}: rd_data = 256'h{hex_data};\n")

# ------------------------------------------------------------
# B1/B2/B3 用 ROM テーブル生成
# ------------------------------------------------------------
def gen_bias_rom_table(layer_name: str,
                       b_i32: np.ndarray,
                       out_ch: int,
                       out_path: Path) -> None:
    """
    rd_data は 1024-bit (32*32) 想定。
    lane0 を rd_data[31:0] に置くため hex は lanes[::-1] で連結。
    """
    b = b_i32

    if layer_name in ("fc2", "fc3"):
        depth = 1
    else:  # fc1
        depth = (out_ch + PE_NUM - 1) // PE_NUM  # 64ch -> 2 words

    with out_path.open("w", encoding="utf-8") as f:
        f.write("// Auto-generated ROM table for %s bias\n" % layer_name)
        f.write("// addr -> rd_data (one 32-lane vector per address)\n\n")

        if layer_name == "fc1":
            # testbench:
            #   for (i group=0..1)
            #     for (j lane=0..31)
            #       b1_wr_data[32*j +:32] = b1[i*PE_NUM + j]
            for addr in range(depth):
                out_base = addr * PE_NUM
                lanes: list[int] = []
                for k in range(PE_NUM):
                    idx = out_base + k
                    if idx < out_ch:
                        v = int(b[idx])
                    else:
                        v = 0
                    lanes.append(v & 0xFFFFFFFF)
                hex_data = "".join(f"{v:08x}" for v in lanes[::-1])
                f.write(f"        {addr}: rd_data = 1024'h{hex_data};\n")

        else:
            # fc2:
            #   for (i=0..31) b2_wr_data[32*i +:32] = b2[i];
            # fc3:
            #   b3_wr_data = 0; for (i=0..9) b3_wr_data[32*i +:32] = b3[i];
            addr = 0
            lanes: list[int] = []
            for k in range(PE_NUM):
                if k < out_ch:
                    v = int(b[k])
                else:
                    v = 0
                lanes.append(v & 0xFFFFFFFF)
            hex_data = "".join(f"{v:08x}" for v in lanes[::-1])
            f.write(f"        {addr}: rd_data = 1024'h{hex_data};\n")

# ------------------------------------------------------------
# まとめて生成
# ------------------------------------------------------------
def gen_all_rom_tables(export_dir: Path, out_dir: Path) -> None:
    meta = load_meta(export_dir / "meta.json")

    fc1_out, fc1_in = map(int, meta["fc1"]["w_shape"])  # 64, 784
    fc2_out, fc2_in = map(int, meta["fc2"]["w_shape"])  # 32, 64
    fc3_out, fc3_in = map(int, meta["fc3"]["w_shape"])  # 10, 32

    # パラメータ読み込み
    W1 = load_weight_rowmajor_int8(export_dir / "fc1_W_rowmajor_int8.hex", fc1_out, fc1_in)
    W2 = load_weight_rowmajor_int8(export_dir / "fc2_W_rowmajor_int8.hex", fc2_out, fc2_in)
    W3 = load_weight_rowmajor_int8(export_dir / "fc3_W_rowmajor_int8.hex", fc3_out, fc3_in)

    b1 = load_bias_int32(export_dir / "fc1_b_int32.hex", fc1_out)
    b2 = load_bias_int32(export_dir / "fc2_b_int32.hex", fc2_out)
    b3 = load_bias_int32(export_dir / "fc3_b_int32.hex", fc3_out)

    out_dir.mkdir(parents=True, exist_ok=True)

    # W* -> 256bit (=8*32) ROM
    gen_weight_rom_table("fc1", W1, fc1_in, fc1_out, out_dir / "W1_rom_table.vh")
    gen_weight_rom_table("fc2", W2, fc2_in, fc2_out, out_dir / "W2_rom_table.vh")
    gen_weight_rom_table("fc3", W3, fc3_in, fc3_out, out_dir / "W3_rom_table.vh")

    # B* -> 1024bit (=32*32) ROM
    gen_bias_rom_table("fc1", b1, fc1_out, out_dir / "B1_rom_table.vh")
    gen_bias_rom_table("fc2", b2, fc2_out, out_dir / "B2_rom_table.vh")
    gen_bias_rom_table("fc3", b3, fc3_out, out_dir / "B3_rom_table.vh")

if __name__ == "__main__":
    EXPORT_DIR = Path(r"../export_hw_fixed_halfup_rowmajor")
    OUT_DIR    = Path("rom_tables")
    gen_all_rom_tables(EXPORT_DIR, OUT_DIR)
    print("ROM tables generated under:", OUT_DIR)


ROM tables generated under: rom_tables
